# Kickoff workspace'u: Unity Catalog + dane źródłowe

Uruchamiasz **raz na nowym workspace** (albo po wyczyszczeniu danych), komórka po komórce, od góry.

| Krok | Co robi | Czas |
|---|---|---|
| 1 | Parametry generowania danych (nazwy katalogów są w `config/platform.yml`) | - |
| 2 | Ścieżki do repo, import kodu, wczytanie `config/platform.yml` | sekundy |
| 3 | Plan SQL (dry-run): co zostanie założone, nic się nie wykonuje | sekundy |
| 4 | Bootstrap UC: katalogi, schemy, volumes | ~1 min |
| 5 | Weryfikacja UC | sekundy |
| 6 | Smoke test: czy zapis Parquet do volume działa | sekundy |
| 7 | Backfill: historia 2025-01-01 → 2026-09-30 do landing | kilka minut |
| 8 | Weryfikacja landing: ile plików per system/tabela | sekundy |

**Wymagania:**
- notebook leży w **głównym folderze repo** (obok `uc_bootstrap.py`), w Git folderze w Workspace,
- compute: **serverless**.

**Wszystko jest bezpieczne do ponownego uruchomienia:**
- bootstrap używa `IF NOT EXISTS`,
- backfill sam się pominie, jeśli landing ma już dane,
- generator jest deterministyczny, więc ponowny run daje identyczne pliki.

Po `Pull` w Git folderze (nowy kod symulatora) zrób **`%restart_python`** albo Detach & re-attach. Python trzyma w pamięci raz zaimportowane moduły i bez restartu nie zobaczy zmian.

## 1. Konfiguracja

Tu są tylko parametry **generowania danych**. Nazwy katalogów, schematów i systemów źródłowych
są w `config/platform.yml` - tym samym pliku, którego używa `uc_bootstrap.py`. Ścieżki do volumes
(`LANDING`, `SMOKE`) notebook wylicza z tego pliku w kroku 2, więc zmiana nazwy projektu to edycja
jednej linii w YAML. Reszta notebooka niczego nie wpisuje na sztywno.

In [0]:
# Zakres historii do wygenerowania.
# 2025-01-01 to "go-live" eksportów: tego dnia CRM wysyła początkowy snapshot klientów.
# Start później = brak klientów sprzed tej daty w danych.
BACKFILL_START = "2025-01-01"
BACKFILL_END = "2026-09-30"

# Skala danych.
#   1.0  -> pełna skala, ~0.9 GB, kilka minut (do nagrań)
#   0.2  -> szybki przebieg, < 1 min (testy, widzowie z ograniczoną quotą)
SCALE = 1.0

# True = generuj backfill nawet wtedy, gdy landing ma już pliki (nadpisze te same pliki).
# Użyj np. po zmianie SCALE. Uwaga: przy zmianie skali najpierw wyczyść landing,
# inaczej zostaną mieszanka starych i nowych plików z dni spoza nowego zakresu.
FORCE_BACKFILL = False

## 2. Ścieżki do repo i import kodu

Notebook uruchomiony z Git folderu ma **katalog roboczy (`os.getcwd()`) = folder, w którym leży notebook**, czyli główny folder repo.
Nie trzeba więc wpisywać e-maila ani nazwy repo: ta sama komórka zadziała na każdym workspace i u każdego widza.

Python importuje moduły tylko z folderów wymienionych w `sys.path`, więc dopisujemy tam:
- `REPO`, żeby zadziałało `import uc_bootstrap`,
- `REPO/retail-simulator`, żeby zadziałało `import simulator`. Pakiet `simulator` jest folderem wewnątrz `retail-simulator`.

Na końcu `load_config()` wczytuje i waliduje `config/platform.yml`, a z niego wyliczamy ścieżki do volumes.
Literówka w YAML zatrzyma notebook już tutaj, a nie w połowie bootstrapu.

`uc_bootstrap` wymaga PyYAML (preinstalowany na Databricks). Jeśli mimo to zobaczysz
`ModuleNotFoundError: No module named 'yaml'`, dodaj na samej górze notebooka komórkę `%pip install pyyaml` i uruchom od początku.

In [0]:
import os
import sys

REPO = os.getcwd()                          # folder notebooka = główny folder repo
SIM = os.path.join(REPO, "retail-simulator")

# Szybka diagnostyka zamiast tajemniczego ModuleNotFoundError niżej.
# Jeśli któryś assert się wywali: notebook nie leży w głównym folderze repo
# albo Git folder nie ma najnowszego commita (zrób Pull).
assert os.path.isfile(os.path.join(REPO, "uc_bootstrap.py")), f"Brak uc_bootstrap.py w {REPO}"
assert os.path.isfile(os.path.join(SIM, "simulator", "cli.py")), f"Brak simulator/cli.py w {SIM}"
assert os.path.isfile(os.path.join(REPO, "config", "platform.yml")), f"Brak config/platform.yml w {REPO}"

# "if not in" chroni przed dopisywaniem tej samej ścieżki przy każdym uruchomieniu komórki.
for path in (REPO, SIM):
    if path not in sys.path:
        sys.path.insert(0, path)

# Obie biblioteki mają funkcję o nazwie main, więc nadajemy im różne nazwy (aliasy).
from uc_bootstrap import main as uc_main      # zakłada katalogi / schemy / volumes
from uc_bootstrap import load_config          # wczytuje config/platform.yml
from simulator.cli import main as sim_main    # generuje pliki źródłowe

# Ten sam plik i ta sama walidacja co w uc_bootstrap.py - nazwy katalogów są w jednym miejscu.
CONFIG = load_config()

# Gdzie symulator pisze pliki. Pod tą ścieżką są volumes per system (erp, crm, ecom, pos)
# założone w kroku 4. Symulator dokłada sam: <system>/<tabela>/load_date=YYYY-MM-DD/<plik>
LANDING = CONFIG.landing_path

# Folder tymczasowy na smoke test (krok 6). Leży w volume, bo Spark na serverless
# NIE czyta lokalnego /tmp drivera (LocalFilesystemAccessDeniedException).
# Na koniec testu folder jest kasowany.
SMOKE = f"{CONFIG.checkpoints_path('dev')}/_smoke"

print("Repo:   ", REPO)
print("Projekt:", CONFIG.project)
print("Landing:", LANDING)

## 3. Plan SQL (dry-run)

`uc_main([...])` przyjmuje te same argumenty co `python uc_bootstrap.py ...` w terminalu, tylko jako listę stringów.
Pusta lista lub lista bez flag to zwykłe uruchomienie. **Zawsze podawaj listę jawnie**: bez niej argparse czytałby `sys.argv`, czyli argumenty procesu notebooka, i mógłby się wywalić.

`--dry-run` tylko **wypisuje** 24 polecenia SQL. Niczego nie wykonuje. To ten sam wynik, który widziałeś lokalnie w terminalu.

In [0]:
uc_main(["--dry-run"])

## 4. Bootstrap Unity Catalog

Wykonuje polecenia z kroku 3 przez `spark.sql`, po kolei: rodzic przed dzieckiem (katalog → schema → volume).
Wynik:
```
<project>_sources.landing            volumes: erp, crm, ecom, pos   ← "świat zewnętrzny", pisze tylko symulator
<project>_dev / <project>_prod       bronze_erp, bronze_crm, bronze_ecom, bronze_pos, silver, gold,
                                     ops (volume: checkpoints)
```
`<project>` to wartość `project` z `config/platform.yml` (np. `moustache_markt`).

Jeśli `CREATE CATALOG` rzuci błąd o *managed location / storage*, załóż ręcznie w **Catalog Explorer** trzy katalogi:
`<project>_sources`, `<project>_dev`, `<project>_prod`. Potem uruchom tę komórkę ponownie. `IF NOT EXISTS` pominie istniejące katalogi i doda resztę.

In [0]:
uc_main([])

## 5. Weryfikacja UC

`display(spark.sql(...))` zamiast komórki `%sql`: komórka `%sql` z dwoma zapytaniami pokazuje wynik tylko ostatniego.

In [0]:
display(spark.sql(f"SHOW VOLUMES IN {CONFIG.sources_catalog}.landing"))   # oczekiwane: crm, ecom, erp, pos
display(spark.sql(f"SHOW SCHEMAS IN {CONFIG.env_catalog('dev')}"))         # oczekiwane: bronze_*, silver, gold, ops (+ default, information_schema)

## 6. Smoke test zapisu Parquet

Generuje 2 dni samego ERP w małej skali (`--systems erp --scale 0.05`) do folderu tymczasowego i czyta go Sparkiem.
Sprawdza dwie rzeczy, zanim puścisz kilkuminutowy backfill:
- czy symulator umie zapisać Parquet (pyarrow),
- czy umie pisać do volume.

Folder `_smoke` leży w `<project>_dev.ops.checkpoints`, a nie w landing, żeby nie zaśmiecić danych źródłowych.
`try/finally` kasuje go także wtedy, gdy odczyt się wywali.

In [0]:
try:
    sim_main(["backfill", "--start", "2025-01-01", "--end", "2025-01-02",
              "--systems", "erp", "--scale", "0.05", "--output", SMOKE])

    # Spark sam rozpozna folder load_date=... jako kolumnę partycji load_date.
    spark.read.parquet(f"{SMOKE}/erp/product").show(5, truncate=False)
finally:
    dbutils.fs.rm(SMOKE, recurse=True)   # sprzątanie, niezależnie od wyniku

## 7. Backfill historii do landing

Generuje wszystkie systemy (ERP, CRM, e-com, POS) za cały zakres z kroku 1.
Co 30 dni symulator wypisuje postęp, a na końcu podsumowanie: wiersze per tabela, liczbę plików, MB.

- **Bez korupcji typów** (`"$ 123"`): ta pojawia się tylko w trybie `daily`. Gdyby była w historii, inferencja schematu w odcinku 1 po cichu zrobiłaby z tych kolumn STRING.
- **Guard:** jeśli landing ma już jakiekolwiek pliki, komórka niczego nie robi, żeby nie palić quoty. Wymuszenie: `FORCE_BACKFILL = True` w kroku 1.

In [0]:
def landing_has_files() -> bool:
    """True, jeśli w którymkolwiek volume w landing jest choć jeden plik/folder."""
    return any(os.listdir(os.path.join(LANDING, system)) for system in os.listdir(LANDING))

if landing_has_files() and not FORCE_BACKFILL:
    print(f"Landing ma już dane - pomijam backfill. Wymuszenie: FORCE_BACKFILL = True w kroku 1.")
else:
    sim_main(["backfill", "--start", BACKFILL_START, "--end", BACKFILL_END,
              "--scale", str(SCALE), "--output", LANDING])

## 8. Weryfikacja landing

Dla każdej tabeli: liczba dni (`load_date=...`), plików, MB i zakres dat.
Czego się spodziewać:
- ERP: 1 plik dziennie dla każdej z 3 tabel,
- POS ma **mniej dni** niż reszta, bo w niedziele niehandlowe nie ma pliku (to celowa pułapka).

`os.walk` przechodzi rekurencyjnie po folderach volume, jak po zwykłym dysku: volumes są zamontowane pod `/Volumes`.

In [0]:
rows = []
for system in sorted(os.listdir(LANDING)):
    for table in sorted(os.listdir(os.path.join(LANDING, system))):
        table_dir = os.path.join(LANDING, system, table)
        days = sorted(d.removeprefix("load_date=") for d in os.listdir(table_dir))
        files = [os.path.join(root, f) for root, _, names in os.walk(table_dir) for f in names]
        rows.append((system, table, len(days), len(files),
                     round(sum(os.path.getsize(f) for f in files) / 1e6, 1),
                     days[0] if days else None, days[-1] if days else None))

display(spark.createDataFrame(rows, "system string, table string, days int, files int, mb double, first_day string, last_day string"))